# Load Silver to ClickHouse + Analytics
=================================

Load silver data to ClickHouse and verify analytics layer views.

In [ ]:
# Configuration
import os
import clickhouse_connect
import boto3
import pandas as pd

CLICKHOUSE_CONFIG = {
    "host": os.getenv("CLICKHOUSE_HOST", "localhost"),
    "port": os.getenv("CLICKHOUSE_PORT", "8123"),
    "database": os.getenv("CLICKHOUSE_DB", "insurance_db"),
    "username": os.getenv("CLICKHOUSE_USER", "default"),
    "password": os.getenv("CLICKHOUSE_PASSWORD", "clickhouse_pass"),
}

MINIO_CONFIG = {
    "endpoint": os.getenv("MINIO_ENDPOINT", "localhost:9900"),
    "access_key": os.getenv("MINIO_ROOT_USER", "minioadmin"),
    "secret_key": os.getenv("MINIO_ROOT_PASSWORD", "minioadmin"),
    "bucket": os.getenv("MINIO_BUCKET", "insurance-data"),
}

ch_client = clickhouse_connect.get_client(
    host=CLICKHOUSE_CONFIG["host"],
    port=CLICKHOUSE_CONFIG["port"],
    database=CLICKHOUSE_CONFIG["database"],
    username=CLICKHOUSE_CONFIG["username"],
    password=CLICKHOUSE_CONFIG["password"],
)

s3 = boto3.client(
    "s3",
    endpoint_url=f"http://{MINIO_CONFIG['endpoint']}",
    aws_access_key_id=MINIO_CONFIG["access_key"],
    aws_secret_access_key=MINIO_CONFIG["secret_key"],
)

print("Connected to ClickHouse and MinIO")

## Current ClickHouse Tables

In [ ]:
# List current tables
result = ch_client.query("SHOW TABLES")
print("=== Tables in ClickHouse ===\n")
for row in result.result_rows:
    print(f"  {row[0]}")

## Load Silver Customers to ClickHouse

In [ ]:
# Find silver customers
def list_objects(prefix: str) -> list:
    response = s3.list_objects_v2(Bucket=MINIO_CONFIG["bucket"], Prefix=prefix)
    return [obj["Key"] for obj in response.get("Contents", [])]

def read_parquet(key: str) -> pd.DataFrame:
    local_path = f"/tmp/{os.path.basename(key)}"
    s3.download_file(MINIO_CONFIG["bucket"], key, local_path)
    df = pd.read_parquet(local_path)
    os.unlink(local_path)
    return df

silver_cust_keys = [k for k in list_objects("silver/silver_customers/") if k.endswith(".parquet")]

if silver_cust_keys:
    # Load latest silver customers
    latest_key = silver_cust_keys[-1]
    df_silver = read_parquet(latest_key)
    print(f"Loaded {len(df_silver)} customers from {latest_key}")
    
    # Drop table if exists and recreate
    ch_client.command("DROP TABLE IF EXISTS silver_customers")
    
    # Create table
    ch_client.command(""""
        CREATE TABLE silver_customers (
            customer_id String,
            first_name String,
            last_name String,
            email String,
            phone String,
            address String,
            city String,
            state String,
            zip_code String,
            credit_score Nullable(Int32),
            risk_bucket String,
            source_system String
        ) ENGINE = MergeTree()
        ORDER BY customer_id
    """)
    
    # Insert data
    ch_client.insert_df("silver_customers", df_silver)
    print(f"✓ Loaded {len(df_silver)} customers to ClickHouse")
else:
    print("No silver customer files found")

## Load Silver Claims to ClickHouse

In [ ]:
# Find silver claims
silver_claim_keys = [k for k in list_objects("silver/silver_claims/") if k.endswith(".parquet")]

if silver_claim_keys:
    # Load latest silver claims
    latest_key = silver_claim_keys[-1]
    df_silver = read_parquet(latest_key)
    print(f"Loaded {len(df_silver)} claims from {latest_key}")
    
    # Drop table if exists and recreate
    ch_client.command("DROP TABLE IF EXISTS silver_claims")
    
    # Create table (simplified schema)
    ch_client.command("""
        CREATE TABLE silver_claims (
            claim_id String,
            customer_id String,
            claim_date Date,
            claim_amount Float32,
            claim_paid_amount Float32,
            claim_status String,
            claim_status_category String,
            vehicle_type String,
            vehicle_category String,
            source_system String
        ) ENGINE = MergeTree()
        ORDER BY (customer_id, claim_date)
    """)
    
    # Insert data
    ch_client.insert_df("silver_claims", df_silver)
    print(f"✓ Loaded {len(df_silver)} claims to ClickHouse")
else:
    print("No silver claim files found")

## Check Analytics Views

In [ ]:
# Check analytics views
print("=== Analytics Views ===\n")

views = [
    "analytics_customers",
    "analytics_claims",
    "analytics_claims_by_status",
    "analytics_claims_by_agent",
    "analytics_claims_by_business_line",
]

for view in views:
    result = ch_client.query(f"SELECT count() FROM {view}")
    count = result.result_rows[0][0] if result.result_rows else 0
    print(f"  {view}: {count} rows")

## Preview Analytics Data

In [ ]:
# Preview risk buckets
result = ch_client.query("""
    SELECT 
        risk_bucket, 
        count() as customers
    FROM analytics_customers
    GROUP BY risk_bucket
    ORDER BY customers DESC
""")
print("=== Risk Buckets ===\n")
print(result.result_rows)

In [ ]:
# Preview claims by status
result = ch_client.query("""
    SELECT 
        claim_status_category, 
        count() as claims,
        sum(claim_amount) as total_amount
    FROM analytics_claims_by_status
    GROUP BY claim_status_category
    ORDER BY total_amount DESC
""")
print("=== Claims by Status ===\n")
print(result.result_rows)

In [ ]:
# Preview claims by business line
result = ch_client.query("""
    SELECT 
        vehicle_category, 
        count() as claims,
        sum(claim_amount) as total_amount
    FROM analytics_claims_by_business_line
    GROUP BY vehicle_category
    ORDER BY total_amount DESC
""")
print("=== Claims by Business Line ===\n")
print(result.result_rows)

## Rebuild Analytics via dbt

In [ ]:
# Run dbt to rebuild analytics layer
import subprocess

result = subprocess.run(
    ["dbt", "run", "--target", "clickhouse"],
    cwd="../dbt",
    capture_output=True,
    text=True,
)

print("=== dbt run output ===")
print(result.stdout)
if result.stderr:
    print("stderr:", result.stderr)

## Quick Validation

In [ ]:
# Quick validation
print("=== Validation ===\n")

# Check silver tables
for table in ["silver_customers", "silver_claims"]:
    result = ch_client.query(f"SELECT count() FROM {table}")
    count = result.result_rows[0][0] if result.result_rows else 0
    print(f"{table}: {count} rows")

# Check analytics views
for view in ["analytics_customers", "analytics_claims"]:
    result = ch_client.query(f"SELECT count() FROM {view}")
    count = result.result_rows[0][0] if result.result_rows else 0
    print(f"{view}: {count} rows")